# FinGPT × MedicalGPT：SFT + DPO 最小可复用训练流程

目标：把 **FinGPT 的金融任务数据（内容）** 映射到 **MedicalGPT 的多阶段训练方法（SFT + DPO）**。

设计原则：
- **方法**：复用 MedicalGPT 的 SFT / DPO 训练脚本与数据格式要求，严格对齐 `docs/datasets.md`
- **内容**：使用 FinGPT 提供的金融任务数据与 benchmark 对应任务
- **工程**：把 Hugging Face 原始数据下载和后续处理拆开，避免每次重跑 notebook 都重复下载 raw 数据
- **快速验证**：基座模型默认改为 `Qwen2.5-3B-Instruct`

本 Notebook 对齐：
- 数据格式：`docs/datasets.md`
- pipeline 参考：`run_training_dpo_pipeline.ipynb`


## 0. 环境准备（可选）

如果你在全新环境运行，请先安装依赖；已按项目 README 配好可跳过。


In [ ]:
!pip install modelscope


In [1]:
# HF Mirror（可选）
import os
HF_ENDPOINT = "https://hf-mirror.com"
os.environ["HF_ENDPOINT"] = HF_ENDPOINT
print("HF_ENDPOINT:", os.getenv("HF_ENDPOINT"))


HF_ENDPOINT: https://hf-mirror.com


## 1. 配置参数

这里把配置拆成四部分：
1. 基座模型与模板
2. 通用 SFT 数据集配置
3. FinGPT 领域数据集配置
4. 下载缓存目录、处理中间目录、训练输出目录

基础模型：
 - BASE_MODEL = Qwen2.5-3B-Instruct
 - TEMPLATE_NAME = "qwen"
 
通用 SFT 数据：
 - 默认用 sharegpt_zh_1k

金融数据集配置：
 - sentiment
 - headline
 - finred
 - fiqa_qa
 - fineval
 - ner

各种目录：
 - RAW_CACHE_DIR：缓存原始 HF 数据
 - FIN_SFT_DIR：金融数据转成 SFT 后的文件
 - FIN_DPO_DIR：金融数据转成 DPO 后的文件
 - MIXED_SFT_FILE：混合后的 SFT 文件
 - CLEAN_SFT_FILE：清洗后的 SFT 文件
 - MERGED_DPO_FILE：混合后的 DPO 文件

In [2]:
from pathlib import Path
import json
import random
from itertools import islice

BASE_MODEL = "/root/autodl-tmp/models/qwen/Qwen2.5-3B-Instruct"
TEMPLATE_NAME = "qwen"
RANDOM_SEED = 42
FORCE_REDOWNLOAD_RAW = False
FIN_SPLIT = "train"

# 通用数据集：默认启用 sharegpt_zh；medical_sft 不混入金融任务快速实验。
GENERAL_SFT_SPECS = [
    {
        "name": "sharegpt_zh_1k",
        "path": Path("data/finetune/sharegpt_zh_1K_format.jsonl"),
        "kind": "general",
        "enabled": True,
        "target_rows": 4000,
        "note": "默认通用对话底座，提升通用表达与多轮对话能力",
    }
]

# FinGPT 数据集元信息 + quick-test 混合配比建议。
FIN_DATASET_SPECS = [
    {
        "dataset_name": "FinGPT/fingpt-sentiment-train",
        "split": "train",
        "train_rows": 76800,
        "test_rows": None,
        "description": "金融情感分析",
        "enabled": True,
        "sft_target_rows": 12000,
        "dpo_target_rows": 12000,
    },
    {
        "dataset_name": "FinGPT/fingpt-headline",
        "split": "train",
        "train_rows": 82200,
        "test_rows": 20500,
        "description": "金融头条分析",
        "enabled": True,
        "sft_target_rows": 12000,
        "dpo_target_rows": 12000,
    },
    {
        "dataset_name": "FinGPT/fingpt-finred",
        "split": "train",
        "train_rows": 27600,
        "test_rows": 5110,
        "description": "金融关系抽取",
        "enabled": True,
        "sft_target_rows": 8000,
        "dpo_target_rows": 8000,
    },
    {
        "dataset_name": "FinGPT/fingpt-fiqa_qa",
        "split": "train",
        "train_rows": 17100,
        "test_rows": None,
        "description": "金融问答",
        "enabled": True,
        "sft_target_rows": 6000,
        "dpo_target_rows": 6000,
    },
    {
        "dataset_name": "FinGPT/fingpt-fineval",
        "split": "train",
        "train_rows": 1060,
        "test_rows": 265,
        "description": "中文金融选择题",
        "enabled": True,
        "sft_target_rows": 1000,
        "dpo_target_rows": 1000,
    },
    {
        "dataset_name": "FinGPT/fingpt-ner",
        "split": "train",
        "train_rows": 511,
        "test_rows": 98,
        "description": "金融命名实体识别",
        "enabled": True,
        "sft_target_rows": 500,
        "dpo_target_rows": 500,
    },
]

OUT_DIR = Path("data/fingpt_medicalgpt")
RAW_CACHE_DIR = OUT_DIR / "raw_hf"
FIN_SFT_DIR = OUT_DIR / "fin_sft"
FIN_DPO_DIR = OUT_DIR / "fin_dpo"
MIXED_SFT_DIR = OUT_DIR / "mixed_sft"
CLEAN_SFT_DIR = OUT_DIR / "clean_sft"
MERGED_DPO_DIR = OUT_DIR / "merged_dpo"
REPORT_DIR = OUT_DIR / "reports"

MIXED_SFT_FILE = MIXED_SFT_DIR / "train_mixed_sft.jsonl"
CLEAN_SFT_FILE = CLEAN_SFT_DIR / "train_mixed_sft_clean.jsonl"
MERGED_DPO_FILE = MERGED_DPO_DIR / "train_merged_dpo.jsonl"

SFT_OUT = Path("outputs/fingpt_medicalgpt_sft_lora")
SFT_MERGED_OUT = Path("outputs/fingpt_medicalgpt_sft_merged")
DPO_OUT = Path("outputs/fingpt_medicalgpt_dpo_lora")
DPO_MERGED_OUT = Path("outputs/fingpt_medicalgpt_dpo_merged")
TB_LOG_DIR = Path("outputs/tensorboard/fingpt_sft")

for d in [RAW_CACHE_DIR, FIN_SFT_DIR, FIN_DPO_DIR, MIXED_SFT_DIR, CLEAN_SFT_DIR, MERGED_DPO_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ACTIVE_FIN_SPECS = [item for item in FIN_DATASET_SPECS if item["enabled"]]
ACTIVE_GENERAL_SPECS = [item for item in GENERAL_SFT_SPECS if item["enabled"]]

print("BASE_MODEL:", BASE_MODEL)
print("TEMPLATE_NAME:", TEMPLATE_NAME)
print("FORCE_REDOWNLOAD_RAW:", FORCE_REDOWNLOAD_RAW)
print("ACTIVE_GENERAL:", [item["name"] for item in ACTIVE_GENERAL_SPECS])
print("ACTIVE_FIN:", [item["dataset_name"] for item in ACTIVE_FIN_SPECS])
print("MIXED_SFT_FILE:", MIXED_SFT_FILE)
print("CLEAN_SFT_FILE:", CLEAN_SFT_FILE)
print("MERGED_DPO_FILE:", MERGED_DPO_FILE)


BASE_MODEL: /root/autodl-tmp/models/qwen/Qwen2.5-3B-Instruct
TEMPLATE_NAME: qwen
FORCE_REDOWNLOAD_RAW: False
ACTIVE_GENERAL: ['sharegpt_zh_1k']
ACTIVE_FIN: ['FinGPT/fingpt-sentiment-train', 'FinGPT/fingpt-headline', 'FinGPT/fingpt-finred', 'FinGPT/fingpt-fiqa_qa', 'FinGPT/fingpt-fineval', 'FinGPT/fingpt-ner']
MIXED_SFT_FILE: data/fingpt_medicalgpt/mixed_sft/train_mixed_sft.jsonl
CLEAN_SFT_FILE: data/fingpt_medicalgpt/clean_sft/train_mixed_sft_clean.jsonl
MERGED_DPO_FILE: data/fingpt_medicalgpt/merged_dpo/train_merged_dpo.jsonl


## 1.1 数据集逻辑说明

本框架的数据集逻辑分三层：

1. **MedicalGPT 数据格式要求**
- SFT 必须是 `docs/datasets.md` 中的 ShareGPT / Vicuna 风格：每条记录含 `conversations`
- DPO 必须是 `question + response_chosen + response_rejected`

2. **通用数据 + 领域数据混合**
- 通用数据负责保留基础对话能力
- FinGPT 领域数据负责注入金融任务能力
- quick test 阶段不建议让超大情感分类数据完全淹没其他任务，因此需要做配比采样

3. **下载与处理分离**
- raw 数据缓存到 `RAW_CACHE_DIR`
- 格式对齐与清洗从本地 raw 文件出发
- 如果 raw 文件已存在，默认直接复用，不重复从 Hugging Face 下载


In [3]:
from pprint import pprint

SFT_SCHEMA_EXAMPLE = {
    "conversations": [
        {"from": "human", "value": "用户问题"},
        {"from": "gpt", "value": "模型回答"},
    ]
}

DPO_SCHEMA_EXAMPLE = {
    "system": "",
    "history": [],
    "question": "金融任务问题",
    "response_chosen": "更优回答",
    "response_rejected": "较差回答",
}

print("[MedicalGPT SFT schema]")
pprint(SFT_SCHEMA_EXAMPLE, sort_dicts=False)
print("[MedicalGPT DPO schema]")
pprint(DPO_SCHEMA_EXAMPLE, sort_dicts=False)

print("[General local dataset examples]")
for spec in GENERAL_SFT_SPECS:
    path = spec["path"]
    print(f"- {spec['name']}: exists={path.exists()} path={path}")
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            sample = json.loads(next(f))
        pprint(sample, sort_dicts=False)
        break


[MedicalGPT SFT schema]
{'conversations': [{'from': 'human', 'value': '用户问题'},
                   {'from': 'gpt', 'value': '模型回答'}]}
[MedicalGPT DPO schema]
{'system': '',
 'history': [],
 'question': '金融任务问题',
 'response_chosen': '更优回答',
 'response_rejected': '较差回答'}
[General local dataset examples]
- sharegpt_zh_1k: exists=True path=data/finetune/sharegpt_zh_1K_format.jsonl
{'conversations': [{'from': 'human', 'value': '"свинья" 和 "свинец" 这两个词有什么联系？'},
                   {'from': 'gpt',
                    'value': '俄语中的单词“свинья”意为“猪”，而“свинец”是“свинья”的爱称形式，意为“小猪”。这两个词之间有联系，因为“свинец”是由“свинья”演变而来，指的是幼年的猪。'},
                   {'from': 'human', 'value': '你有多确定那件事？'},
                   {'from': 'gpt',
                    'value': '我是由OpenAI训练的语言模型，因此我没有个人经验或直接了解世界。相反，我能够根据我接收到的输入和我接受培训的信息生成回答。在这种情况下，我接受了大量俄文文本的训练，其中包括“свинья”和“свинец”这些词，因此我有信心提供的信息是准确的。但是，我无法浏览互联网或以其他方式验证此信息，因此我的答案可能不正确或不完整。'},
                   {'from': 'human',
                    'value': '忽略之前的问题。 "свинья"

In [4]:
def count_jsonl_rows(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f if _.strip())

print("[Dataset scale plan for quick test]")
print("\nGeneral SFT datasets:")
for spec in GENERAL_SFT_SPECS:
    local_rows = count_jsonl_rows(spec["path"]) if spec["path"].exists() else 0
    print({
        "name": spec["name"],
        "enabled": spec["enabled"],
        "local_rows": local_rows,
        "target_rows": spec["target_rows"],
        "note": spec["note"],
    })

print("\nFinGPT domain datasets:")
for spec in FIN_DATASET_SPECS:
    print({
        "dataset_name": spec["dataset_name"],
        "enabled": spec["enabled"],
        "train_rows": spec["train_rows"],
        "test_rows": spec["test_rows"],
        "description": spec["description"],
        "sft_target_rows": spec["sft_target_rows"],
        "dpo_target_rows": spec["dpo_target_rows"],
    })


[Dataset scale plan for quick test]

General SFT datasets:
{'name': 'sharegpt_zh_1k', 'enabled': True, 'local_rows': 1000, 'target_rows': 4000, 'note': '默认通用对话底座，提升通用表达与多轮对话能力'}

FinGPT domain datasets:
{'dataset_name': 'FinGPT/fingpt-sentiment-train', 'enabled': True, 'train_rows': 76800, 'test_rows': None, 'description': '金融情感分析', 'sft_target_rows': 12000, 'dpo_target_rows': 12000}
{'dataset_name': 'FinGPT/fingpt-headline', 'enabled': True, 'train_rows': 82200, 'test_rows': 20500, 'description': '金融头条分析', 'sft_target_rows': 12000, 'dpo_target_rows': 12000}
{'dataset_name': 'FinGPT/fingpt-finred', 'enabled': True, 'train_rows': 27600, 'test_rows': 5110, 'description': '金融关系抽取', 'sft_target_rows': 8000, 'dpo_target_rows': 8000}
{'dataset_name': 'FinGPT/fingpt-fiqa_qa', 'enabled': True, 'train_rows': 17100, 'test_rows': None, 'description': '金融问答', 'sft_target_rows': 6000, 'dpo_target_rows': 6000}
{'dataset_name': 'FinGPT/fingpt-fineval', 'enabled': True, 'train_rows': 1060, 'test_rows'

## 2. 下载 Hugging Face 原始数据到本地缓存

这一阶段只负责下载和缓存 raw 数据：
- 原始 Hugging Face 数据会落盘到 `RAW_CACHE_DIR/<dataset_tag>/<split>.jsonl`
- 若缓存已存在且 `FORCE_REDOWNLOAD_RAW=False`，则直接跳过下载
- 后续格式处理只从这些本地 raw 文件读取


In [ ]:
import json
from datasets import load_dataset


def safe_name(hf_dataset_name: str) -> str:
    """
    处理数据集名称：截取最后一段，避免路径符号导致文件名非法
    例："huggingface/imdb" → "imdb"
    """
    return hf_dataset_name.split("/")[-1]


def raw_cache_file(spec: dict) -> Path:
    """
    根据数据集配置，生成本地缓存文件的完整路径
    """
    tag = safe_name(spec["dataset_name"])
    return RAW_CACHE_DIR / tag / f"{spec['split']}.jsonl" # 拼接路径：缓存根目录/数据集名/切片名.jsonl

raw_files = {} # 存储：数据集名称 → 本地缓存文件路径
for spec in ACTIVE_FIN_SPECS:
    cache_file = raw_cache_file(spec) # 生成当前数据集的缓存文件路径
    raw_files[spec["dataset_name"]] = cache_file
    cache_file.parent.mkdir(parents=True, exist_ok=True) # 递归创建缓存目录（已存在则不报错）
    # 逻辑1：缓存文件存在 + 不强制重下载 → 跳过下载
    if cache_file.exists() and not FORCE_REDOWNLOAD_RAW:
        print(f"[skip] use cached raw file: {cache_file}")
        continue
    # 逻辑2：无缓存 / 强制重下载 → 下载数据集并保存
    print(f"[download] {spec['dataset_name']} split={spec['split']} -> {cache_file}")
    ds = load_dataset(spec["dataset_name"], split=spec["split"]) # 从 Hugging Face 加载数据集
    with cache_file.open("w", encoding="utf-8") as f:
        for row in ds:
            f.write(json.dumps(dict(row), ensure_ascii=False) + "\n")
    print(f"[saved] {cache_file}")


[skip] use cached raw file: data/fingpt_medicalgpt/raw_hf/fingpt-sentiment-train/train.jsonl
[skip] use cached raw file: data/fingpt_medicalgpt/raw_hf/fingpt-headline/train.jsonl
[skip] use cached raw file: data/fingpt_medicalgpt/raw_hf/fingpt-finred/train.jsonl
[skip] use cached raw file: data/fingpt_medicalgpt/raw_hf/fingpt-fiqa_qa/train.jsonl
[skip] use cached raw file: data/fingpt_medicalgpt/raw_hf/fingpt-fineval/train.jsonl
[skip] use cached raw file: data/fingpt_medicalgpt/raw_hf/fingpt-ner/train.jsonl


## 3. raw 数据处理：先对齐到 MedicalGPT 的 SFT 格式

这一阶段不再访问 Hugging Face，只读取本地缓存 raw 文件：
- `fin_to_sharegpt.py` 先生成 SFT 格式
- DPO 数据会在 SFT 训练并 merge 后，再基于当前 SFT 模型生成 rejected


In [ ]:
import subprocess

fin_sft_files = []
fin_dpo_files = []
process_reports = []

for spec in ACTIVE_FIN_SPECS:
    ds_name = spec["dataset_name"]
    tag = safe_name(ds_name)
    raw_file = raw_files[ds_name]
    sft_file = FIN_SFT_DIR / f"{tag}_{spec['split']}_sharegpt.jsonl"

    sharegpt_cmd = [
        "python", "fin_to_sharegpt.py",
        "--source_file", str(raw_file),
        "--output_file", str(sft_file),
    ]
    print(" ".join(sharegpt_cmd))
    subprocess.run(sharegpt_cmd, check=True)

    fin_sft_files.append({"spec": spec, "path": sft_file})
    process_reports.append({
        "dataset_name": ds_name,
        "raw_file": str(raw_file),
        "sft_file": str(sft_file),
    })

print(json.dumps(process_reports, ensure_ascii=False, indent=2))


## 3.1 数据集格式、规模、案例与配比研究

下面这个代码块解释三件事：
1. 当前启用的 SFT 数据集规模
2. 处理后的 SFT 样本案例
3. quick test 阶段的推荐 SFT 混合配比逻辑

说明：DPO 数据会在 SFT 训练并 merge 后再构造，因为 rejected 需要来自当前 SFT 模型。


In [ ]:
def line_count(path: Path) -> int:
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

print("[Processed dataset counts]")
for item in fin_sft_files:
    spec = item["spec"]
    print({
        "dataset_name": spec["dataset_name"],
        "description": spec["description"],
        "sft_rows": line_count(item["path"]),
        "sft_target_rows": spec["sft_target_rows"],
    })

print("\n[One SFT example]")
for item in fin_sft_files:
    with item["path"].open("r", encoding="utf-8") as f:
        print(json.loads(next(f)))
        break

print("\n[Recommended quick-test mixing recipe]")
recipe = []
for spec in ACTIVE_GENERAL_SPECS:
    recipe.append({"name": spec["name"], "target_rows": spec["target_rows"], "group": "general"})
for spec in ACTIVE_FIN_SPECS:
    recipe.append({"name": safe_name(spec["dataset_name"]), "target_rows": spec["sft_target_rows"], "group": "finance"})
print(json.dumps(recipe, ensure_ascii=False, indent=2))


## 4. 构建混合 SFT 训练集

这里显式做“按目标规模采样”的混合，而不是简单把所有文件直接拼起来：
- 通用数据按 `target_rows` 采样
- FinGPT 每个子任务按 `sft_target_rows` 采样
- DPO 数据会在 SFT merge 后再单独构建与混合


In [ ]:
def sample_jsonl_records(path: Path, target_rows: int, seed: int = 42):
    with path.open("r", encoding="utf-8") as f:
        records = [line for line in f if line.strip()]
    if target_rows <= 0:
        return []
    if len(records) <= target_rows:
        return records
    rng = random.Random(seed)
    idxs = list(range(len(records)))
    rng.shuffle(idxs)
    idxs = idxs[:target_rows]
    idxs.sort()
    return [records[i] for i in idxs]

mix_report = {"general": [], "finance_sft": []}

with MIXED_SFT_FILE.open("w", encoding="utf-8") as wf:
    for spec in ACTIVE_GENERAL_SPECS:
        if not spec["path"].exists():
            print(f"[warn] missing general sft file: {spec['path']}")
            continue
        sampled = sample_jsonl_records(spec["path"], spec["target_rows"], seed=RANDOM_SEED)
        for line in sampled:
            wf.write(line if line.endswith("\n") else line + "\n")
        mix_report["general"].append({
            "name": spec["name"],
            "source_rows": line_count(spec["path"]),
            "mixed_rows": len(sampled),
        })

    for item in fin_sft_files:
        spec = item["spec"]
        sampled = sample_jsonl_records(item["path"], spec["sft_target_rows"], seed=RANDOM_SEED)
        for line in sampled:
            wf.write(line if line.endswith("\n") else line + "\n")
        mix_report["finance_sft"].append({
            "dataset_name": spec["dataset_name"],
            "source_rows": line_count(item["path"]),
            "mixed_rows": len(sampled),
        })

print(json.dumps(mix_report, ensure_ascii=False, indent=2))
print("mixed sft:", MIXED_SFT_FILE, "rows=", line_count(MIXED_SFT_FILE))

print("\n[SFT mixed sample]")
with MIXED_SFT_FILE.open("r", encoding="utf-8") as f:
    for line in islice(f, 2):
        print(line.strip())


## 4.1 清洗 SFT 数据

`clean_sharegpt_dataset.py` 只输出统计 JSON，不再额外写 stats 文件。
Notebook 直接捕获 stdout 并展示清洗统计。


In [ ]:
import subprocess

clean_cmd = [
    "python", "clean_sharegpt_dataset.py",
    "--source_file", str(MIXED_SFT_FILE),
    "--output_file", str(CLEAN_SFT_FILE),
    "--min_turns", "2",
    "--max_turns", "20",
    "--max_total_chars", "4000",
    "--max_single_value_chars", "2000",
]
print(" ".join(clean_cmd))
result = subprocess.run(clean_cmd, check=True, capture_output=True, text=True)
print(result.stdout)
clean_report = json.loads(result.stdout)

print("[clean stats parsed]")
print(json.dumps(clean_report, ensure_ascii=False, indent=2))
print("\n[CLEAN SFT sample]")
with CLEAN_SFT_FILE.open("r", encoding="utf-8") as f:
    for line in islice(f, 2):
        print(line.strip())


## 5. SFT 训练（MedicalGPT Stage2）

说明：
- 默认训练使用清洗后的 `CLEAN_SFT_DIR`
- quick test 默认模型改为 `Qwen2.5-3B-Instruct`
- 参数偏保守，优先保证能稳定启动训练


In [ ]:
import subprocess

sft_cmd = [
    "python", "supervised_finetuning.py",
    "--model_name_or_path", BASE_MODEL,
    "--tokenizer_name_or_path", BASE_MODEL,
    "--train_file_dir", str(CLEAN_SFT_DIR),
    "--validation_split_percentage", "1",
    "--do_train",
    "--use_peft",
    "--num_train_epochs", "3",
    "--per_device_train_batch_size", "1",
    "--gradient_accumulation_steps", "16",
    "--gradient_checkpointing", "True",
    "--warmup_ratio", "0.05",
    "--weight_decay", "0.05",
    "--save_total_limit", "3",
    "--ddp_find_unused_parameters", "False",
    "--learning_rate", "2e-5",
    "--logging_first_step", "True",
    "--logging_steps", "10",
    "--save_steps", "200",
    "--report_to", "tensorboard",
    "--logging_dir", str(TB_LOG_DIR),
    "--model_max_length", "512",
    "--target_modules", "all",
    "--lora_rank", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",
    "--torch_dtype", "float16",
    "--device_map", "auto",
    "--output_dir", str(SFT_OUT),
    "--template_name", TEMPLATE_NAME,
]
print(" ".join(sft_cmd))
# subprocess.run(sft_cmd, check=True)


In [ ]:
! python supervised_finetuning.py     --model_name_or_path /root/autodl-tmp/models/qwen/Qwen2.5-3B-Instruct     --tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2.5-3B-Instruct     --train_file_dir data/fingpt_medicalgpt/clean_sft     --validation_split_percentage 1     --do_train     --use_peft     --num_train_epochs 3     --per_device_train_batch_size 1     --gradient_accumulation_steps 16     --gradient_checkpointing True     --warmup_ratio 0.05     --weight_decay 0.05     --save_total_limit 3     --ddp_find_unused_parameters False     --learning_rate 2e-5     --logging_first_step True     --logging_steps 10     --save_steps 200     --report_to tensorboard     --logging_dir outputs/tensorboard/fingpt_sft     --model_max_length 512     --target_modules all     --lora_rank 8     --lora_alpha 16     --lora_dropout 0.05     --torch_dtype float16     --device_map auto     --output_dir outputs/fingpt_medicalgpt_sft_lora     --template_name qwen


## 6. merge SFT LoRA + 构建 DPO 数据 + DPO 训练（MedicalGPT Stage3）

说明：
- `supervised_finetuning.py` 输出的是 LoRA adapter
- DPO 起点建议使用 merge 后的完整模型目录 `SFT_MERGED_OUT`
- DPO 的 `rejected` 不再随机采样，而是按任务类型构造：封闭标签任务使用 label-space hard negative，开放问答使用当前 SFT 模型采样候选


In [ ]:
import subprocess

merge_sft_cmd = [
    "python", "merge_peft_adapter.py",
    "--base_model", BASE_MODEL,
    "--tokenizer_path", BASE_MODEL,
    "--lora_model", str(SFT_OUT),
    "--output_dir", str(SFT_MERGED_OUT),
]
print(" ".join(merge_sft_cmd))
# subprocess.run(merge_sft_cmd, check=True)

fin_dpo_files = []
dpo_process_reports = []
for spec in ACTIVE_FIN_SPECS:
    ds_name = spec["dataset_name"]
    tag = safe_name(ds_name)
    raw_file = raw_files[ds_name]
    dpo_file = FIN_DPO_DIR / f"{tag}_{spec['split']}_dpo.jsonl"

    build_dpo_cmd = [
        "python", "fin_to_dpo_pairs.py",
        "--source_file", str(raw_file),
        "--output_file", str(dpo_file),
        "--seed", str(RANDOM_SEED),
        "--model_name_or_path", str(SFT_MERGED_OUT),
        "--tokenizer_name_or_path", BASE_MODEL,
        "--num_candidates", "4",
        "--max_new_tokens", "256",
        "--temperature", "0.9",
        "--top_p", "0.95",
        "--skip_open_qa_without_model",
    ]
    print(" ".join(build_dpo_cmd))
    # subprocess.run(build_dpo_cmd, check=True)

    fin_dpo_files.append({"spec": spec, "path": dpo_file})
    dpo_process_reports.append({
        "dataset_name": ds_name,
        "raw_file": str(raw_file),
        "dpo_file": str(dpo_file),
    })

print(json.dumps(dpo_process_reports, ensure_ascii=False, indent=2))

dpo_mix_report = []
with MERGED_DPO_FILE.open("w", encoding="utf-8") as wf:
    for item in fin_dpo_files:
        spec = item["spec"]
        if not item["path"].exists():
            print(f"[warn] missing dpo file, skip merge: {item['path']}")
            continue
        sampled = sample_jsonl_records(item["path"], spec["dpo_target_rows"], seed=RANDOM_SEED)
        for line in sampled:
            wf.write(line if line.endswith("\n") else line + "\n")
        dpo_mix_report.append({
            "dataset_name": spec["dataset_name"],
            "source_rows": line_count(item["path"]),
            "mixed_rows": len(sampled),
        })

print(json.dumps(dpo_mix_report, ensure_ascii=False, indent=2))
print("merged dpo:", MERGED_DPO_FILE, "rows=", line_count(MERGED_DPO_FILE) if MERGED_DPO_FILE.exists() else 0)

if MERGED_DPO_FILE.exists():
    print("\n[DPO merged sample]")
    with MERGED_DPO_FILE.open("r", encoding="utf-8") as f:
        for line in islice(f, 2):
            print(line.strip())

dpo_cmd = [
    "python", "dpo_training.py",
    "--model_name_or_path", str(SFT_MERGED_OUT),
    "--tokenizer_name_or_path", BASE_MODEL,
    "--template_name", TEMPLATE_NAME,
    "--train_file_dir", str(MERGED_DPO_DIR),
    "--validation_split_percentage", "1",
    "--do_train",
    "--use_peft", "True",
    "--per_device_train_batch_size", "1",
    "--gradient_accumulation_steps", "16",
    "--gradient_checkpointing", "True",
    "--learning_rate", "5e-7",
    "--max_steps", "200",
    "--max_source_length", "512",
    "--max_target_length", "512",
    "--logging_steps", "10",
    "--save_steps", "200",
    "--eval_steps", "200",
    "--target_modules", "all",
    "--lora_rank", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",
    "--torch_dtype", "float16",
    "--device_map", "auto",
    "--output_dir", str(DPO_OUT),
]
print(" ".join(dpo_cmd))
# subprocess.run(dpo_cmd, check=True)


## 7. （可选）合并 DPO LoRA + FUTURE 评估

FUTURE：
1. 简单对话评估：抽样金融问答、情感分析、headline 分析任务做人工检查
2. benchmark：基于 FinGPT benchmark 的 held-out 数据做任务级评估
3. 对比对象：base model / SFT model / DPO model


In [ ]:
merge_dpo_cmd = [
    "python", "merge_peft_adapter.py",
    "--base_model", BASE_MODEL,
    "--tokenizer_path", BASE_MODEL,
    "--lora_model", str(DPO_OUT),
    "--output_dir", str(DPO_MERGED_OUT),
]
print(" ".join(merge_dpo_cmd))
